# Lista 5 — Questão 07: Apriori e Titanic

Resolução comentada e reproduzível. Execute as células em ordem. Dependências: Python 3, pandas e numpy. O algoritmo Apriori está implementado abaixo, sem depender de mlxtend.

**Nota sobre o material:** o Apriori.ipynb da disciplina não estava entre os arquivos fornecidos. Esta é uma implementação independente do mesmo algoritmo.

**Base:** [seaborn-data/titanic.csv](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv), 891 passageiros. Uma cópia compactada está incorporada para execução sem internet.

In [1]:
from pathlib import Path
from itertools import combinations
import json
import math
import hashlib
import pandas as pd
import numpy as np

ROOT = Path.cwd()
URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'



## 1. Cópia reprodutível da base
Se o CSV não estiver ao lado do notebook, restauramos a cópia utilizada na resolução. O hash evita trocar silenciosamente a base.

In [2]:
import base64, gzip
snapshot = 'H4sIABf30moC/71d2Y7dxhF9z7dcEKxe2OSjZ6zxs6C8B7I0iYTIkqElTv4+3Hov3q6qHhsGDCMZ8pLs7lpOnTr17cfX/3z8z/P72+/vPr399u327fm/t7f/er59+/jrt99vv7/9+u7D7Z9vvz7fnn/79e3Xf69/ePzdHx++3N6+//Hp+z9+e/vp+fb++d2/zz/5x/cvf3y+vf203nX995fPz38bb/q2/5VSw3iD23hzg7K3N7e/f/j49f36f32+/f3rj+fb7c2XH98/vP3t9+9fPt8+f7k9vf307flvsF7xz+f9ej3762FQs9a3x9vTx6/fvt/++LLdZP/79X97/PD89dcvP77+6/a/52/hLtrfRU3rXcb9KZbkMdJ7ZE+y3WR7vuxJ7PkkVg+w3gJ7jPIex4OEj7HfYvtnfSnCx9ifIFx8XmjsrG+vq0tf/3h+/vxtX4d4JRxXWnP+rIVhnvb3Px4+XP7q/k9va6jXuykYRhcf/N2Hj5/eY58vWci4BG5/CHUDGECvC0laBH8b5W8D5lwFPa7PMs7bR39+9+Vz8TSN/XDcZP1nGhz6Or9cPknYD3Y+v+q6uawlbojiu44d+0Ev+0vYm15PhiOfrTF+BTDhVMzWKNK6nqcirIf1Wxq2I/bGL0b7ZGWby2ybaxlg3ZuvsYfId3fcFPFggE5/Hv8G2W/7Uw3nfoJ5v0FjT2I7+/yEan34R+z6fC+eD6AKi6Cm9vOnn/+42i/gxdv/fGXU4hZILJJaVPPrZ4bxWD5/DNZXQSzLT80VmF/AuJxeYj8LenaWY1xyA5uvY1yHuIiVeYVlfwO1ruGxDMUneKTuoNklK5A9ObYC5XPPi53JNuR8dOMN0GqdndqtafHwyIsnNvA4OWY1gOnF6aM/tEzx+fTOct783P+Td+sw7nuPdHrGbOduLzCrAVz75f128d/ORzZWYat+vej5ZluX/N5uQ+wlyJyGP/HbW6sugxd88OrKlQEXXqDpgpPn2LfedpNlMJQDmyyAymKK7R77N2k7n8wJVmeHcugTz6f331Y3A4N1+yoKApHddvSfflE8ua/gZrRb0eTLnFn/s+pYr8k1dj4asfgjC+v3Yu2Z89fdGWyswdM0s7xNff7SR2g57GAxzeLzmTX0PA5/bTR/vtg2ceP7uyh61FWYr+nw/HCbYFhc8iDhVR6wxQh+/47Pf7wKO+LzA2q4OVHjvF5ItKFY2HoeXrXv4jfo0W2EC7C9g90MwDQsoo2kwkZivEKeGe+Hf7OkqOtNv5v3Wj6LnfVpd2nRSnhq44Mdx3zrdN/wwo1osGAzWMo0jPX1kbm/5Z4aey4Y63mALQunHv5ovg7zozgwRLlpptN9z8M0KUbW51MGJQqZ4itM2K6nee3CfGreA8TXP4LOwWw5K8mDhC0U3t5Og2nHy4jZseHMTsjHfyJAOMxIXejmQ444DutO3b7+fgA4li7Nr3yeAIoWrSFfzn/6JQk72ycnP7vLYKlLhoQrMy9gKRNc4xLcr/310zMvcXXp59e70d3DtVlmN+A4OHrDDDQPLRLuv+ColEbzYxZyqkwHUqaWnotHHCSjZtjeYK1hFjgK9IoZ/D1UWfePYOFszDCU/bOtTnht5638ut2mw05XwBAe3cQtswenetCbq60ub4bIJzYGx714IXIKrwEdnKvSlWA6Cpt/z12+GLZzOlq4uXXlZ2zr/dzYOlqHczNxtr6/3J2hSh7tUDfu1ccjR1tJojbxaj75x3fOsuME3fvwx74za6BJclklNuWdDo5NPTajPLP6KizMusA27iQ3MpRKRXRoVpaZ6Vee1xxVswY6X5t+QKuGxO3vxpChOkut2Klbkd8Dy/JA5i1X27duBIsmWg8NpEWdmUpRXKIkemmgr+iBfmfAkWY4Fq8QNnKE6DrUuXJw6TpeXZ3/vOTKN77B+IDqAVkvYLq7uw+qYGsNeLdfFkTcmydfTYC+OnZP95/c+DRrTTIdvzKqc+NLq+bU8Z7EcBkXE1TLgtdfBNxTRZpKKYvW1sd/v606eIAt909QGfGfQMkW8R+kjTcoyNlAKLXDCBdkoAqmkDQqmBw7fAqHYL2+aT/LbXAwHQqs6trv1GyJiA06buzibxGqNPKkcQnHkO3CwEdAq/2y7C0Iy8nQ0FMG0FLLa8rF78fEec7Pt+zeS5X5MsMBro9u/SnADuFT43qjeg6xhQjZWGXp3ALI9uCeuk7DJEvbrT09qSTzNuP5/TJTyoof3KBBsZNXC7E0otHSSOP0QqSbuY3qxK+S6bGfKUbDh4tL5/V7T8tBaOJ9cONDnvXtgc70U1lB+PQ6TlgUSr47P2wI+YLs04G4tBisVnCb43lkG3dAeWn7Guy0LOzIvrpzZP1CHCUik5d4KKlXVuKxZ9JEwUvrpJ8G0FdHdwq4k8Yqkw8NiyOmosWj75mFBcuRU1kU5p3eevlDsNqQ6Qh/CdBb3Mbz+QZUvLO0uzuXyw5mosRccef4wmxgJM2Dg52KdZ/vW+d/o8ht6uC0j6tHcpkiDftq88kojYWY29Ii7jG1G2eyrrmmR0XrpQ7rdVElemqXiYy3X2vkwLNfue2wI4Za/UTCzDJGSxsyg6wwXtN5qYDfSWcCDqOm3nRMtDBFLMYGYnH1/KoI9TmnpiQyQV0mIbK7E0YVFPEybQ9Ggo25oDTeYyWWoW/gdrO4jSh0T0+cdBb2w8b1H+kGuJ9SmeaNjL107cOX4a+MPwMZmhk5RMjJMhnAkPlMYrSO5Zp+xWFc3eaF4fyl9foKTffJCXOg0s5HoZxESr3IeRgbDyoq8s7IvxuzIIG/4YN2qfkMth+kTQxKUmV9kYjltLqOzuOEuvlh/WXlLIuMWHb28JurcqQDDps9MsAeqPyfmwa17GB5+0WuFoFBjIIKbORZAFWgVaINYKEXK2CGHXDLW/OWsV1kxNKcyAjSbFoTLIzlQjuvrPi8RmiTuVzeP2MpEqVEUPciFixOHqdmEaElquCVgOadl4SBy8pZsDKH6SMznm9PzNYKmyUihyROZ05qJFwKsuQQ1G5HR4RYOdH3wygCjCqHJGHLa+SSxS8dNvCxPj0m1G1mu9i+cZQ3m69rs3mVKnp3a9Fj29y5UIX4AVmnVxihKM7BxjCxR8J3t8CNLKAV06uyDioJu8RXl8qAO/0CvzRQzkndBwweCVsIjsZZEDTvGohgsQJlZa1cykPOxFrlFeJclElb/a+pKwuVjmnAHdkDrTX/aHBXg77qqsEfRBXII0jK7p2EC93jCz3sqzxuuxx0NyrPtMy/0GDugehKCalf7YemHkO+R95Y3YWc+dqAXNGYfldKABv2p709ocdj2D6G9b8mfUGceGxwj2IWTSWrl80NbewbO0DGl01hYXqVnPIxHhUYV4MAj614QoY/nFdP2m+ENZE9+ns4rJW8fkbnblTnIVkC3lFw/eWrM7GA3avISCOTxYAgGvaZcn4FjOOE8mO5PSawiDL4rOQ50csWKMl64gKPwXkyGmOwEJhZdeDTU1CCkhIjhpWsz3xgnojTb8UOoXCxbPoTF6HDwwVdsfJgajiw01qJ4y5dLGlvma2gTSDBPyWtAuyezPjDmt+Gm1R71dnbAhbSGDq1WI/XYEji7UZWHzLUMV9JlP6zNSwg74fXg2rYynv0vJHRUqeyh97i5e2HUYrzi+oOhAUbh0XxFv066YA1ZJyvVMEaiQcEpu44DwtDVyyHjM8+jZFMM68zHzutn+TC8rxqvEW0HPpoSqa8R/Up5pPsqNakVl/UAa5tYGqNgZ5AvVTLT4TkkgKy4pFGa504ehUbRVf6+NtBsW2XWuFTN7UHB2Ayw3zAzKKMJNIa9AYVUXboHZUDfvOl6yVGJI3LV4buwj6rggcLRQc1Cxhgl7XTxZzKrJB8xmNQDqE4rrSTlEinJiugrTamIWYXlpdepNZqOp22dblyCc9n136X02Dn2QK7DgkZ+4OsOxABDHjdDZWME31THZ5vTT4mYtx8RziJ21mxYLzytlwaFEDDyDBKecxlbE+uF67G1W/uZ03qJThOynR01KvC18pahLoajC6MCdkYEShKT224ahRy5qoCVgrgk0yZzvluYClSOlibnOpirIc+r2WvYkmSRm15/cVRdPKgGGdJI7252GIyYNzCEblwJ+e1ld00Qmq+qjSfmNy8Xq3JF9OqPOPJoreAKl4UOyMDVkHJuWpsFthV61rD2f0FmntowVmw+SafqTo7KCHNC4N52un2C2HzweKvQbTmUXpzi7GePmMAqS000pigPEhll15hNRI1FaWuMoCW4Uq0KoUau3nEWEqlEsrEGN7LpAnCLZcQV3tPKa08VDd2r1sJuLYfJFypNGQNZAHl9tSHDlhlrUVnqd7JBAEiTZLRWIRK4KsLH/ZnpA0l604kHCcUa+RlwI2Ci99AqdB2gyCAyIipFGuk4J3x6XfUVREK/EiwmjSlOZFy3ixRAyplxPrJyoH6lEr/X/N+7iNajtdokKNzdJSxRjongWw8ou/AojqrStZq3YsTW9MuDi+gWqI7jb5CdSHoUBfKtcNZ3RalwBGSutH3QFQnAwl9zgLGUqAOHvCZ6yyWz03tiROsoZBE/WJanNO67tRJLKkL1Slp/jWK3NznvXLpIVA06AvOXBrCZyeOQzrVfujKLq/EI52qrFK1m2+5qt/Yx96H0UehuzqbQPhZrk0EuK4EkVyTas1LrKbvDN9yZ8mRC63lJWX1z6L8lj5LVJRLd9+Cul1BCMhmiN6X03t1zXTIPTaJq/yCSKfKEiBhUbTU5+BR9j0/dowIwEKmx6ZlJ4NG0Om5f2iTA09VvEpXiKcQdrQQ6NL/svT/jUfiWFNrohmFviDSWLxYzxJDH+WdxizaL7oIc0f9IPWnAIqhag23TGphhiMOwbbjTwSLplHOLlFxJu+7ZgknFJ1g6ngEleoFEuaopNx3Ng8sL0ch0yDIbmm3jcuwMCVP8nDk+PEWqFXsAg+Iz8sAIyaz/EiQiGSxuJBgYiHMYyhkehpiG69aSNRhyYtxhhSedt3w0TGEhO3N/bLNnNdH1MFMB5gctIE1Sl141UAe5q5edfGww1lSdRET6kspJYppu2qtdooR7J1bc9z2lMgmdrXHFXNlOEXlNNHTuDnlBYwRpNKDO1r6Uc7RBeuprDksw6x53RiZIJHiMPtfBqBRUVFnF2Fl66dHmHJnNvP1031ooLyEMZ+hsEhnJPX2wliBgGLy7lOkRtjZcScG5wFm2ViAasiiycILCgVkGg00opA3gsFZoT1ZD/cACy1chbB3uugxsrisxloEyaZlBKTXwzPW8M6iUR0JJBLLgZR6+UTV59DP7eZBbZ+cXqApft9SOqKrXhKLN5Ncb3Zk0Y20tNOpIFqPrGB10pYsi0kR2DlYZcoxzR4m/j0JG7rjQN0Z7cxotYHmAUgtvMyRN+hQAoxYm8pIztTEapKNKMMyM0foKXqhrh6dhMr4wINX7W4sb0DsMhg6xQQ7vRLt2JgYrfc5IqbWzAa0xtE3ZPD4iNngFypHQE/dGKsW9oTWTU2L5p7+rrZUKbuzVo6U9BWM3LHOV2LnPDjhJaCMzOCt/sdtb84WM1lEisUpH0EdoslWIqXUJtWTZ79z1w0czmWmSsV7vVxNGFCE5ghaSiRIvp/rkKIiyuc+kA0HjKvjYzDkqthVTs5We8TN6icqu7TUEb3TDF+1mIt86lKsT4P0JpANiaUqM2EpY9A2KtayRdVMa1PXigrNkZtLT6032Uv6wJfvsswRdFbjXW4EexaE4ZDRJ5yaDPgTXTVINwsDpSQlj2FWMBwYTEdsKDZfhTbl5jkRybgl69Vk+vrkZdMjQk3JTxdfPvfEOU2YOhN8QUWgKF0maZvmFNtrjEgDRqx9PXY1gKclWfuXFCfkzKg8a1Eh6Bc3FrDD5VwCzpTNAXwuHWOKPZJutPV+si6+/LCyVGDTlFOfH9/CYNLIl9onXyW/mpf89om2C9SHr1sbWKETkvKyaRCl2Sq4OcyxNVGMlDTq82oyPIscmeYgns+iRwHoVNrvu8ImDddtjch1l3t5tcWAeaC702POXeiAn34Haiffd8XqtEsmF3J0M9IuOSCM/LvXFtAxsktQGLcCUlxqAFU/aG+cTOIjTYGpgHfJQnNdGrI6SNZwfG8poqSlDUFmEQ0bkuJcOWWoVB26HLVwRT+EQq7/L5KPOIyT2iNTFh/DGEkxEopzdqQF9+JyJCOcYreIZLLJKAVl81QOL0a2PZtXaTMw2AMXbXef4UpCh2lmiaRWJapj/9oS3uCHqxJegEyxvDMr0laycUs9KnWw7mRlAYNR/xndThMmds3iPBvxgLKYHnYJDatksnMmHdDUe8sJ8GzBdMjozicjIusAvrJC15QGibpquJymWH41mHFT6KYX12JU63AlL2rjVXx5lrrv5YDRjtoUvXcn7J55jDa8yUSrnztp9hvpKx+dX8avbgbGiL8eyThugYbrg0co7NkWc/T7WubzJgm+ImEt828reXOOv5OMGcgbLFgWo1+3o5iNJtoE6ZkTcCm6mbg52xuO4VQOjbgb+ZUM4oN6IKwdLHeyWj0ih84f1ZGAyoPiO2ZTlZ2icDyGrL0oMulMOsuXHi0iZRASqpe/w9ZsLBmsEJBlomLGi3RIpxPZqnmo9MQ+pX8LiPuhTbjQirgS5C4bikavrpgrJLYroHUhgz8gKJTerGfhEXs06nGyhnN28jCTDoWWmIySbbs+acEixOKPfzdaBg9kHUmKybIu9QihynBY4zMA0+e55mAUX96NwpbOoiGUMw+gn+kfdYEkq67MybuS9QlA1ASct4yM5+X8yh+EgSnvkqDx/NMDD/X8I0avwdmMqhmepoaYyCFONY5RokTpd4Dp7DKZ8KPHslgoQNaUJYzPj40wYmhKpKMAd4U27jecJcz3tHzjlRU3cTatnWV1yENtwhxXpiobQr8plg8GJscJ2DtShrEaoiTw/NMo6jdURRe0iP0gQ8er8nUKbRBC3bwoAUS+Ur39TUzTJ+Tpn5q7N8iD3NHne7yf56UlAhYFvo74O/RJ0/GQm0diasRMIfajJfxl7LcIas+lKkS9hxltd0b1NyJEvc3rVqKWtsOZ+2RDFe/jtBhsUnaAERT/CR0sj/cBgwBY4GEkfbDwMpiZnDejdkwQDi59lkDPBKXOx8YsyTjQ+5KP/4oCfpCkgi/mifGUSsOj+1AyL2pzuPRhsA4VbC5tiOnRlrejSFseiiKP1Ttc2jQdV4CpuHl+3PHOsmmcKCtfOmMem0wVjctsTkVOhSoBBFbjcrAhVTjJ6SWCkJPOgzsms7Wk5nuz0j78qu/q2IFo+YU6XTCYtUSiqgyF6AlpL/TbhRoXs+RGSnEWoc1BNYqQMdUkKINJhnClXGJutSdyIEEmEAkJfLTT54CFPqUk1nr0KJ2LpUdxsQ9ueeMIbv1/bvF0gD35tCA4QFWvIHV9+CFPUbDDXKAITQemdYekmu6c1nVaHgFTJaCQ47Cm/CdTRapMO8tEAEqdcmI3Y7+0UykvK2u+F3bRxRQcREzUutLcbCNAKeS4vjm1/y6n0QtF/i10DZDOnFDaPEQd8oC2EFPUH2IBg9Z4ftU0Pg9aMobIyE2OEQ6Mjp3T3Eytn4irCr0D3VGiTmFbK6UjIQv+1BAos5GMZajSUBdcKI56XlowQbFmynlPyzcatzp3sOLLMbIZYE2zO1DgJZTQEzu3attBhk8sVj2zjFKgu6dyVhSpGfyQuWPefRgKsn+99exzBc6gmMTQ7hnhnL+7vA6YhL25Hh6op5FS2GRFkjQdsp8cZSrb1d0ZsGUusuOtxhJ5FXvnA4MBCsj0jnmYUnSTQypkBDu4o8e6Vihr2AOxpakWiMWbY486iPW0wwEeB7OzdNa1sLJaX0Bqp2Fx/EEkWjbFAoodPXLkJTET6pxohEjniQrDLLJ8mdq3WTGqZ34IE+tNtAau4v2tsCEgwcrFch15zC5R8r3eOg/Npm+NQj2MreMtwNaBd1SoWJyHwLaaSIFLAeuOUlJw2fcIHJ2qK9NBwvoRyHC+d/qb7FqHoz2809dPPPHGk1unYpAsKqqZn4RX7T0mx4OhkJcO0wzp7rZw6M7B5/fZiwYiwlMHdGvWvYOXxFJXKbwnaB+CqCutBcCWZ4jMq6s5ePjtqd/XfG7JuGSxkG3AR9htCFktH5XffaTx6kTdKzCJ+KC1pNfuLNh6XtpTM9evpp0V6LLJ1zrhE1n+RqnzK3Krr6WPVbiSvylr9/QRi+goCYryhI5DSfriIWf6psIlmtEZ86Y1WrzWr92domFCuCrTkmRIkGHSgIELtmwNj6yxS1BN2YHJDPOhg02e2hrbIUA0r6Ieu6mrNiq6uqtcWhN8M8eaI4yz5o8dhzB5lKVPmWuCCZGHftxGFTC7MOc3Siz0k/Oi4JTWO7YBVUogb3LYvenovSk5cY4baRnYbjz0wMMAaTsYq7Y4derrGq83WraRs2pMoUxoedlj3p+0kFVL0EF8hjjIuDaMOMW6Bdpj3QazEai8Ly/RBC5bfuud070w+LEp4BK4+nKhaN0rO5dKtwFxvsWV3J9Iz0FY/EgTkr3tJK8+MFocXR/6BmjZiFjrjxFfWTxi0LyCNdufgJmbJJ311ArI/wGFXbiVut4AAA=='
if not (ROOT / 'titanic.csv').exists():
    (ROOT / 'titanic.csv').write_bytes(gzip.decompress(base64.b64decode(snapshot)))
assert hashlib.sha256((ROOT / 'titanic.csv').read_bytes()).hexdigest() == '81787d320d7f7b03df935e91de8bd19e11d45c5bbcab86ef4d4a76dc91b7d4f2'
print('Base conferida: 891 passageiros.')

Base conferida: 891 passageiros.


## 2. Tratamento e discretização
177 idades ausentes: mediana por sexo e classe, com mediana global de reserva; o alvo não participa da imputação. Dois portos ausentes: moda S. Idade: [0,18), [18,23), [23,41), [41,+∞); tarifa: [0,10), [10,30), [30,100), [100,+∞). Familia = sibsp + parch, sem o próprio passageiro, em 0, 1–3 e 4+. As fronteiras foram escolhidas antes da mineração. Removemos campos redundantes, derivados e deck (688 ausentes). alive replica o alvo e causaria vazamento.

In [3]:
def preparar():
    bruto = pd.read_csv(ROOT / 'titanic.csv')
    # Não usar o desfecho na imputação: mediana por sexo e classe.
    idade = bruto.age.fillna(bruto.groupby(['sex', 'pclass']).age.transform('median'))
    idade = idade.fillna(bruto.age.median())
    dados = pd.DataFrame({
        'Classe': bruto.pclass.map({1: '1a', 2: '2a', 3: '3a'}),
        'Sexo': bruto.sex.map({'female': 'Feminino', 'male': 'Masculino'}),
        'Idade': pd.cut(idade, [-np.inf, 18, 23, 41, np.inf], right=False,
                        labels=['0-17', '18-22', '23-40', '41+']),
        'Tarifa': pd.cut(bruto.fare.fillna(bruto.fare.median()),
                         [-np.inf, 10, 30, 100, np.inf], right=False,
                         labels=['<10', '10-<30', '30-<100', '100+']),
        'Embarque': bruto.embarked.fillna(bruto.embarked.mode()[0]),
        'Familia': pd.cut(bruto.sibsp + bruto.parch, [-1, 0, 3, np.inf],
                          labels=['Sozinho', '1-3', '4+']),
    })
    positivo = pd.get_dummies(dados, prefix_sep='=', dtype=bool)
    negativo = (~positivo).rename(columns=lambda s: 'NÃO ' + s)
    alvo = pd.DataFrame({'Sobreviveu=Sim': bruto.survived.eq(1),
                         'Sobreviveu=Não': bruto.survived.eq(0)})
    # Não duplicar o alvo por negação: NÃO Sim já corresponde a Não.
    binario = pd.concat([positivo, negativo, alvo], axis=1)
    return bruto, dados, binario


bruto, dados, binario = preparar()
print('Dimensões:', binario.shape)
print('Ausentes na matriz:', int(binario.isna().sum().sum()))
print(dados.head().to_string(index=False))
print('\nExemplos de itens:', ', '.join(binario.columns[:8]))

Dimensões: (891, 40)
Ausentes na matriz: 0
Classe      Sexo Idade  Tarifa Embarque Familia
    3a Masculino 18-22     <10        S     1-3
    1a  Feminino 23-40 30-<100        C     1-3
    3a  Feminino 23-40     <10        S Sozinho
    1a  Feminino 23-40 30-<100        S     1-3
    3a Masculino 23-40     <10        S Sozinho

Exemplos de itens: Classe=1a, Classe=2a, Classe=3a, Sexo=Feminino, Sexo=Masculino, Idade=0-17, Idade=18-22, Idade=23-40


## 3. Presença e ausência
Para cada indicador x, NÃO x = 1 − x. São 19 indicadores de perfil, 19 complementos e dois indicadores do alvo: 40 colunas. As negações são feitas após tratar os ausentes. NÃO Classe=1a significa classe 2a ou 3a; não significa dado faltante. Não criamos negações extras do alvo.

In [4]:
perfil = [x for x in binario if not x.startswith(('NÃO ', 'Sobreviveu='))]
assert all((binario[x] != binario['NÃO ' + x]).all() for x in perfil)
print('Complementos verificados:', len(perfil))
print(bruto.groupby(['sex', 'pclass']).age.median().to_string())

Complementos verificados: 19
sex     pclass
female  1         35.0
        2         28.0
        3         21.5
male    1         40.0
        2         30.0
        3         25.0


## 4. Apriori
Junção de itemsets frequentes e poda: um candidato só é contado se todos os subconjuntos de tamanho k−1 forem frequentes. Limitamos max_len=3 para apresentar antecedentes com até dois itens e um alvo. A busca não cobre regras mais longas.

In [5]:
def apriori(binario, suporte=0.05, max_len=3):
    """Apriori: junção de prefixos e poda de subconjuntos infrequentes."""
    matriz = binario.to_numpy(dtype=bool)
    n = len(matriz)
    minimo = math.ceil(suporte * n - 1e-10)
    contagens = {}
    nivel = set()
    for j in range(matriz.shape[1]):
        qtd = int(matriz[:, j].sum())
        if qtd >= minimo:
            nivel.add((j,))
            contagens[(j,)] = qtd
    for k in range(2, max_len + 1):
        candidatos = set()
        ordenado = sorted(nivel)
        for i, a in enumerate(ordenado):
            for b in ordenado[i + 1:]:
                if a[:-1] != b[:-1]:
                    continue
                candidato = a + (b[-1],)
                if all(s in nivel for s in combinations(candidato, k - 1)):
                    candidatos.add(candidato)
        proximo = set()
        for c in candidatos:
            qtd = int(matriz[:, c].all(axis=1).sum())
            if qtd >= minimo:
                proximo.add(c)
                contagens[c] = qtd
        nivel = proximo
        if not nivel:
            break
    return contagens



## 5. Métricas e filtros
Suporte = n(A e B)/N; confiança = n(A e B)/n(A); lift = confiança/P(B). Geramos primeiro todas as partições dos itemsets frequentes. Depois mantemos apenas consequente unitário de sobrevivência, sem o alvo no antecedente, atributos distintos no antecedente e lift > 1. Antecedentes com a mesma máscara de passageiros são apresentados uma vez, dando preferência a menos negações. É uma redução empírica; não equivale a uma prova de equivalência lógica ou significância estatística.

In [6]:
def regras(binario, contagens, suporte, confianca):
    nomes = list(binario.columns)
    n = len(binario)
    todas = []
    for conjunto, qtd in contagens.items():
        if len(conjunto) < 2 or qtd / n + 1e-12 < suporte:
            continue
        for k in range(1, len(conjunto)):
            for a in combinations(conjunto, k):
                b = tuple(x for x in conjunto if x not in a)
                conf = qtd / contagens[a]
                if conf + 1e-12 >= confianca:
                    todas.append(dict(a=[nomes[x] for x in a], b=[nomes[x] for x in b],
                                      n=qtd, na=contagens[a], suporte=qtd / n,
                                      confianca=conf, lift=conf / (contagens[b] / n)))
    # Regra de interesse: apenas sobrevivência no consequente, sem alvo no antecedente.
    uteis = []
    for r in todas:
        if len(r['b']) != 1 or not r['b'][0].startswith('Sobreviveu='):
            continue
        if any(x.startswith('Sobreviveu=') for x in r['a']):
            continue
        atributos = [x.removeprefix('NÃO ').split('=')[0] for x in r['a']]
        if len(set(atributos)) != len(atributos) or r['lift'] <= 1 + 1e-12:
            continue
        uteis.append(r)
    # Antecedentes equivalentes na amostra são apresentados uma única vez.
    uteis.sort(key=lambda r: (sum(x.startswith('NÃO ') for x in r['a']), len(r['a']), r['a']))
    vistos = set()
    unicas = []
    for r in uteis:
        chave = (np.packbits(binario[r['a']].all(axis=1).to_numpy()).tobytes(), r['b'][0])
        if chave not in vistos:
            vistos.add(chave)
            unicas.append(r)
    unicas.sort(key=lambda r: (-r['lift'], -r['confianca'], -r['suporte'], r['a']))
    return todas, unicas


def executar():
    bruto, dados, binario = preparar()
    frequentes = apriori(binario)
    configuracoes = []
    escolhidas = None
    for sup, conf in [(0.05, 0.70), (0.10, 0.70), (0.10, 0.80), (0.15, 0.80), (0.10, 0.90)]:
        todas, uteis = regras(binario, frequentes, sup, conf)
        configuracoes.append(dict(suporte=sup, confianca=conf,
                                  itemsets=sum(v / len(bruto) + 1e-12 >= sup for v in frequentes.values()),
                                  regras=len(todas), uteis=len(uteis)))
        if (sup, conf) == (0.10, 0.80):
            escolhidas = uteis
            pd.DataFrame(todas).to_json(ROOT / 'regras_completas.json', orient='records', force_ascii=False, indent=2)
    resultado = dict(n=len(bruto), sobreviventes=int(bruto.survived.sum()),
                     ausentes=bruto.isna().sum().to_dict(), colunas=binario.shape[1],
                     sha256=hashlib.sha256((ROOT / 'titanic.csv').read_bytes()).hexdigest(),
                     configuracoes=configuracoes, regras=escolhidas)
    (ROOT / 'resultados.json').write_text(json.dumps(resultado, ensure_ascii=False, indent=2), encoding='utf-8')
    dados.to_csv(ROOT / 'dados_tratados.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame(escolhidas).to_csv(ROOT / 'regras_selecionadas.csv', index=False, encoding='utf-8-sig')
    print(json.dumps(resultado, ensure_ascii=False, indent=2))
    return resultado



In [7]:
resultado = executar()
print('\nComparação:')
print(pd.DataFrame(resultado['configuracoes']).to_string(index=False))

{
  "n": 891,
  "sobreviventes": 342,
  "ausentes": {
    "survived": 0,
    "pclass": 0,
    "sex": 0,
    "age": 177,
    "sibsp": 0,
    "parch": 0,
    "fare": 0,
    "embarked": 2,
    "class": 0,
    "who": 0,
    "adult_male": 0,
    "deck": 688,
    "embark_town": 2,
    "alive": 0,
    "alone": 0
  },
  "colunas": 40,
  "sha256": "81787d320d7f7b03df935e91de8bd19e11d45c5bbcab86ef4d4a76dc91b7d4f2",
  "configuracoes": [
    {
      "suporte": 0.05,
      "confianca": 0.7,
      "itemsets": 6188,
      "regras": 9475,
      "uteis": 131
    },
    {
      "suporte": 0.1,
      "confianca": 0.7,
      "itemsets": 4344,
      "regras": 7690,
      "uteis": 108
    },
    {
      "suporte": 0.1,
      "confianca": 0.8,
      "itemsets": 4344,
      "regras": 5055,
      "uteis": 36
    },
    {
      "suporte": 0.15,
      "confianca": 0.8,
      "itemsets": 3144,
      "regras": 4247,
      "uteis": 31
    },
    {
      "suporte": 0.1,
      "confianca": 0.9,
      "itemsets": 4344

## 6. Regras ordenadas por lift
Configuração escolhida: suporte 0,10 e confiança 0,80. A tabela apresenta todas as 36 regras após os filtros. Percentuais são calculados sobre a base tratada.

In [8]:
tabela = pd.DataFrame(resultado['regras'])
tabela['antecedente'] = tabela.a.map(lambda a: ' E '.join(a))
tabela['consequente'] = tabela.b.map(lambda b: b[0])
print(tabela[['antecedente','consequente','n','na','suporte','confianca','lift']].to_string(index=False))

                        antecedente    consequente   n  na  suporte  confianca     lift
          Classe=1a E Sexo=Feminino Sobreviveu=Sim  91  94 0.102132   0.968085 2.522116
      Sexo=Feminino E NÃO Classe=3a Sobreviveu=Sim 161 170 0.180696   0.947059 2.467337
     Sexo=Feminino E NÃO Embarque=S Sobreviveu=Sim  91 109 0.102132   0.834862 2.175036
        Sexo=Feminino E Idade=23-40 Sobreviveu=Sim 104 128 0.116723   0.812500 2.116776
        Sexo=Feminino E Familia=1-3 Sobreviveu=Sim 125 155 0.140292   0.806452 2.101019
        Sexo=Masculino E Tarifa=<10 Sobreviveu=Não 243 272 0.272727   0.893382 1.449916
           Idade=23-40 E Tarifa=<10 Sobreviveu=Não 167 191 0.187430   0.874346 1.419020
         Classe=3a E Sexo=Masculino Sobreviveu=Não 300 347 0.336700   0.864553 1.403128
     Sexo=Masculino E NÃO Classe=1a Sobreviveu=Não 391 455 0.438833   0.859341 1.394668
       Tarifa=<10 E NÃO Idade=18-22 Sobreviveu=Não 205 240 0.230079   0.854167 1.386270
   Sexo=Masculino E NÃO Familia=

## 7. Interpretação
1. Mulheres da 1a classe: 91/94 sobreviveram; confiança 96,81%, lift 2,522.
2. Mulheres NÃO da 3a classe: 161/170; confiança 94,71%, lift 2,467.
3. Mulheres que NÃO embarcaram em S: 91/109; confiança 83,49%, lift 2,175.
4. Homens com tarifa <10: 243/272 não sobreviveram; confiança 89,34%, lift 1,450.
5. Homens da 3a classe: 300/347 não sobreviveram; confiança 86,46%, lift 1,403.
6. Tarifa <10 e NÃO idade 18–22: 205/240 não sobreviveram; confiança 85,42%, lift 1,386.

As negações agrupam categorias e não devem ser interpretadas como causas. Classe, sexo, porto e tarifa podem estar associados entre si. As idades imputadas podem mudar de faixa e afetar as regras; os resultados são descritivos desta amostra, sem validação externa.

## 8. Conferência independente das métricas
Recontamos diretamente no CSV binarizado cada uma das regras publicadas, sem consultar as contagens do Apriori.

In [9]:
for r in resultado['regras']:
    a = binario[r['a']].all(axis=1)
    b = binario[r['b']].all(axis=1)
    n_ab = int((a & b).sum())
    assert n_ab == r['n'] and int(a.sum()) == r['na']
    assert abs((a & b).mean() - r['suporte']) < 1e-12
    assert abs(n_ab / a.sum() - r['confianca']) < 1e-12
    assert abs((n_ab / a.sum()) / b.mean() - r['lift']) < 1e-12
assert len(resultado['regras']) == 36
print('36 regras conferidas por contagem direta. Nenhum valor ausente na matriz.')

36 regras conferidas por contagem direta. Nenhum valor ausente na matriz.
